
# Yelp Dataset — Exploration Notebook

This notebook helps you quickly **peek, profile, and sample** the Yelp JSON files without loading them fully into memory.

**Files expected (Yelp Academic Dataset):**
- `business.json` (JSON Lines)
- `review.json` (JSON Lines)
- `user.json` (JSON Lines)
- `tip.json` (JSON Lines)
- `checkin.json` (JSON Lines)

> You can change `DATA_DIR` below to match your local path.


In [2]:
pip install pandas

Defaulting to user installation because normal site-packages is not writeable
  Using cached pandas-2.3.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
Using cached pandas-2.3.3-cp313-cp313-win_amd64.whl (11.0 MB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)

   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   ---------------------------------------- 0/2 [pytz]
   -----

In [3]:

from pathlib import Path
import json
import os
from typing import Iterator, Dict, Any, List, Optional
from collections import Counter, defaultdict
import itertools
import pandas as pd

DATA_DIR = Path("./Yelp JSON/yelp_dataset")
FILES = {
    "business": DATA_DIR / "yelp_academic_dataset_business.json",
    "review":   DATA_DIR / "yelp_academic_dataset_review.json",
    "user":     DATA_DIR / "yelp_academic_dataset_user.json",
    "tip":      DATA_DIR / "yelp_academic_dataset_tip.json",
    "checkin":  DATA_DIR / "yelp_academic_dataset_checkin.json",
}

def is_jsonl(path: Path, max_lines: int = 3) -> bool:
    """Heuristically check JSON Lines by reading a few lines."""
    try:
        with path.open("r", encoding="utf-8") as f:
            for i, line in zip(range(max_lines), f):
                line = line.strip()
                if not line:
                    continue
                json.loads(line)
            return True
    except Exception:
        return False

def iter_jsonl(path: Path) -> Iterator[dict]:
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def head_jsonl(path: Path, n: int = 5) -> List[dict]:
    return list(itertools.islice(iter_jsonl(path), n))

def count_lines(path: Path, chunk_size: int = 1 << 20) -> int:
    """Fast-ish line count for large files."""
    with path.open("rb") as f:
        return sum(buf.count(b"\n") for buf in iter(lambda: f.read(chunk_size), b""))

def schema_keys_sample(path: Path, sample: int = 10000) -> Counter:
    """Collect key frequency over a sample of records."""
    c = Counter()
    for i, rec in enumerate(iter_jsonl(path)):
        c.update(rec.keys())
        if i + 1 >= sample:
            break
    return c

def to_parquet_sample(path: Path, out_parquet: Path, max_rows: int = 500_000, dtype_overrides: Optional[dict] = None):
    """Stream a JSONL file into a Parquet sample with optional dtype controls."""
    rows = []
    for i, rec in enumerate(iter_jsonl(path)):
        rows.append(rec)
        if (i + 1) % 50_000 == 0:
            df = pd.DataFrame(rows)
            if dtype_overrides:
                for col, dt in dtype_overrides.items():
                    if col in df.columns:
                        df[col] = df[col].astype(dt, errors='ignore')
            df.to_parquet(out_parquet, engine="pyarrow", compression="zstd", index=False, append=out_parquet.exists())
            rows = []
        if i + 1 >= max_rows:
            break
    if rows:
        df = pd.DataFrame(rows)
        if dtype_overrides:
            for col, dt in dtype_overrides.items():
                if col in df.columns:
                    df[col] = df[col].astype(dt, errors='ignore')
        df.to_parquet(out_parquet, engine="pyarrow", compression="zstd", index=False, append=out_parquet.exists())
    return out_parquet


## 1) Quick file overview

In [4]:

overview = []
for name, path in FILES.items():
    exists = path.exists()
    size_mb = path.stat().st_size / (1024*1024) if exists else None
    kind = "jsonl" if (exists and is_jsonl(path)) else ("missing" if not exists else "unknown")
    n = count_lines(path) if exists else 0
    overview.append(dict(file=name, path=str(path), exists=exists, size_mb=size_mb, n_lines=n, format=kind))
pd.DataFrame(overview)


,file,path,exists,size_mb,n_lines,format
0,business,Yelp JSON\yelp_dataset\yelp_academic_dataset_b...,True,113.357348,150346,jsonl
1,review,Yelp JSON\yelp_dataset\yelp_academic_dataset_r...,True,5094.403108,6990280,jsonl
2,user,Yelp JSON\yelp_dataset\yelp_academic_dataset_u...,True,3207.520495,1987897,jsonl
3,tip,Yelp JSON\yelp_dataset\yelp_academic_dataset_t...,True,172.237849,908915,jsonl
4,checkin,Yelp JSON\yelp_dataset\yelp_academic_dataset_c...,True,273.665376,131930,jsonl


## 2) Peek at first records (heads)

In [5]:

heads = {name: head_jsonl(path, 3) for name, path in FILES.items() if path.exists()}
for name, rows in heads.items():
    print(f"\n### {name}.json : {len(rows)} rows shown\n")
    for r in rows:
        print(json.dumps(r, ensure_ascii=False)[:1200])
        print("---")



### business.json : 3 rows shown

{"business_id": "Pns2l4eNsfO8kk83dixA6A", "name": "Abby Rappoport, LAC, CMQ", "address": "1616 Chapala St, Ste 2", "city": "Santa Barbara", "state": "CA", "postal_code": "93101", "latitude": 34.4266787, "longitude": -119.7111968, "stars": 5.0, "review_count": 7, "is_open": 0, "attributes": {"ByAppointmentOnly": "True"}, "categories": "Doctors, Traditional Chinese Medicine, Naturopathic/Holistic, Acupuncture, Health & Medical, Nutritionists", "hours": null}
---
{"business_id": "mpf3x-BjTdTEA3yCZrAYPw", "name": "The UPS Store", "address": "87 Grasso Plaza Shopping Center", "city": "Affton", "state": "MO", "postal_code": "63123", "latitude": 38.551126, "longitude": -90.335695, "stars": 3.0, "review_count": 15, "is_open": 1, "attributes": {"BusinessAcceptsCreditCards": "True"}, "categories": "Shipping Centers, Local Services, Notaries, Mailbox Centers, Printing Services", "hours": {"Monday": "0:0-0:0", "Tuesday": "8:0-18:30", "Wednesday": "8:0-18:30", "Th

## 3) Schema key frequencies (sample 10k)

In [6]:

key_freq = {}
for name, path in FILES.items():
    if path.exists():
        key_freq[name] = schema_keys_sample(path, sample=10_000)
frames = []
for name, freq in key_freq.items():
    df = pd.DataFrame(freq.items(), columns=["key", "count"]).sort_values("count", ascending=False)
    df.insert(0, "file", name)
    frames.append(df)
keys_df = pd.concat(frames, ignore_index=True)
import caas_jupyter_tools
caas_jupyter_tools.display_dataframe_to_user("Yelp key frequencies (sampled)", keys_df)


ModuleNotFoundError: No module named 'caas_jupyter_tools'

## 4) Basic counts and selected distributions

In [7]:

# Count unique business_id, user_id in sampled chunks (approximate)
def unique_ids(path: Path, field: str, max_rows: int = 1_000_000):
    s = set()
    for i, rec in enumerate(iter_jsonl(path)):
        if field in rec:
            s.add(rec[field])
        if i + 1 >= max_rows:
            break
    return len(s)

stats = {}
if FILES["business"].exists():
    stats["n_business_unique"] = unique_ids(FILES["business"], "business_id", 1_000_000)
if FILES["user"].exists():
    stats["n_user_unique"] = unique_ids(FILES["user"], "user_id", 1_000_000)
if FILES["review"].exists():
    stats["n_review_lines"] = count_lines(FILES["review"])
stats


{'n_business_unique': 150346,
 'n_user_unique': 1000000,
 'n_review_lines': 6990280}

## 5) Create Parquet samples (fast iteration)

In [8]:

SAMPLES_DIR = DATA_DIR / "_samples"
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

out_paths = {}
for name in ("business", "review", "user"):
    src = FILES[name]
    if src.exists():
        out = SAMPLES_DIR / f"{name}_sample.parquet"
        out_paths[name] = to_parquet_sample(src, out, max_rows=300_000)
out_paths


ImportError: Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.

## 6) Example: join review sample → business sample

In [9]:

# Load samples (adjust if large for your machine)
b = pd.read_parquet(SAMPLES_DIR / "business_sample.parquet")
r = pd.read_parquet(SAMPLES_DIR / "review_sample.parquet")

# Select minimal columns to keep memory low
b_small = b[["business_id", "name", "categories", "city", "state", "stars"]].copy()
r_small = r[["business_id", "user_id", "stars", "date"]].copy()

# Merge and show a small head
merged = r_small.merge(b_small, on="business_id", how="left")
merged.head(10)


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

## 7) Plots (run if desired)

In [ ]:

# Matplotlib-only plots (run manually if you want visuals)
import matplotlib.pyplot as plt

if 'b' in globals():
    # Distribution of business stars (sample)
    plt.figure()
    b['stars'].dropna().astype(float).hist(bins=20)
    plt.title('Business stars (sample)')
    plt.xlabel('stars')
    plt.ylabel('count')
    plt.show()



## 8) Next steps (suggested)
- Validate JSON structure variability (nested fields like `attributes`, `hours`, `categories`).
- Build **city/region subsets** for quicker iteration.
- Normalize/flatten nested dicts (e.g., `attributes`) for EDA (consider `pd.json_normalize` on streamed samples).
- Create **interaction matrix** (user × business) for ELSA pretraining (implicit: 1 if user reviewed/checked in).
- Persist tidy parquet tables: `business.parquet`, `users.parquet`, `interactions.parquet` (row = (user_id, business_id, ts)).
- Draft evaluation splits (strong generalization or temporal split).
- Prototype ELSA → embed users → train **TopK-SAE** on those embeddings for interpretability.
